# 🚀 ReduCNN: High-Level Research Orchestrator

This notebook demonstrates the most streamlined way to use the `reducnn` framework. 

Instead of manually managing training loops, mask builders, and surgical operations, we use the **`Orchestrator`**. This high-level API is designed for researchers who want to focus on **experiment results** rather than implementation details.

### Workflow Demonstrated:
1. **Define Configuration**: Specify the model, pruning ratio, and target criterion.
2. **Load Data**: Prepare standard PyTorch/Keras data loaders.
3. **Run Pipeline**: The `Orchestrator` handles baseline loading, pruning surgery, fine-tuning, and automatic checkpointing.

In [ ]:
import os
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from reducnn.engine.orchestrator import Orchestrator
from reducnn.visualization import PruningVisualizer

print("✅ Framework Ready.")

### Step 1: Experiment Configuration
We define a `run_id` (naming convention) to ensure all exports are organized without overwriting previous runs. The `Orchestrator` will use these paths for automatic saving.

In [ ]:
RUN_ID = "resnet18_cifar10_chip_v1"

config = {
    'model_type': 'resnet18',
    'backend': 'pytorch',
    'method': 'chip',       # Activation-based Importance
    'scope': 'global',     # Network-wide ranking
    'ratio': 0.4,          # 40% filter removal
    'epochs': 0,           # Set to 0 because we will load a pretrained model
    'ft_epochs': 2,        # Healing phase
    'experiment_id': RUN_ID,
    'pruned_checkpoint_path': f'checkpoints/{RUN_ID}/pruned.pth',
    'final_checkpoint_path': f'checkpoints/{RUN_ID}/final_finetuned.pth'
}

print(f"📌 Experiment Configured: {RUN_ID}")

### Step 2: Data & Pretrained Model
We load a standard CIFAR-10 dataset and a pretrained ResNet-18 model. 

> **Note**: In a real scenario, you would provide the path to your actual `.pth` or `.h5` file. Here, we'll download one or use a freshly initialized one for the demo.

In [ ]:
# Standard Transform
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load Dataset (CIFAR-10)
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

# Load a Pretrained ResNet-18 from torchvision for this experiment
from reducnn.backends.factory import get_adapter
adapter = get_adapter(None, config)
pretrained_model = adapter.get_model('resnet18', pretrained=True)

print("✅ Dataset and Pretrained Model Loaded.")

### Step 3: Run the Pipeline
We pass the model and loaders to the `Orchestrator`. It will:
1. Evaluate the baseline performance.
2. Perform **Structural Pruning** using the `CHIP` method.
3. **Save** the pruned intermediate model to `checkpoints/{RUN_ID}/pruned.pth`.
4. Fine-tune for 2 epochs.
5. **Save** the final model to `checkpoints/{RUN_ID}/final_finetuned.pth`.
6. Generate ROI and Sensitivity plots.

In [ ]:
orchestrator = Orchestrator(config)

# The 'Real Saving Logic' is handled internally by the Orchestrator calling the Backend Adapter.
pruned_model, masks = orchestrator.run(
    model=pretrained_model, 
    loader=trainloader, 
    val_loader=testloader
)

### Step 4: Advanced Visual Reporting
Using our new **`PruningVisualizer`** with the same `RUN_ID` to ensure naming consistency in the `outputs/` folder.

In [ ]:
pvis = PruningVisualizer(model_name="ResNet18", framework="PyTorch", experiment_id=RUN_ID)

# Visualize the macro structure change (Compression Animation)
from reducnn.visualization import LayerVisData

# Prepare metadata for the structural report
before_layers = []
after_layers = []

for name, mask in masks.items():
    n_orig = len(mask)
    n_pruned = int(np.sum(mask == 0)) # mask is 1 for keep, 0 for prune (Wait, check mask builder)
    # Note: mask builder uses 1.0 for keep, 0.0 for prune
    before_layers.append({"layer_name": name, "num_channels": n_orig})
    after_layers.append({"layer_name": name, "num_channels": n_pruned})

pvis.animate_structure_change(before_layers, after_layers, filename="structural_ROI.gif")
pvis.display_inline(f"outputs/{RUN_ID}/structure/structural_ROI.gif")

print(f"✅ All exports saved under: outputs/{RUN_ID}/")